In [3]:
# ============================================================================
# COMPREHENSIVE UNIT TESTING FOR MOE PHISHING DETECTION SYSTEM
# ============================================================================

import unittest
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import joblib
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.base import BaseEstimator, TransformerMixin
from scipy.sparse import csr_matrix
import re

# ============================================================================
# IMPORT YOUR CLASSES (Copy from original code)
# ============================================================================

class URLFeatures(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    
    def transform(self, urls):
        urls = np.array(urls).reshape(-1)
        feats = np.array([
            [
                len(u),
                u.count('-'),
                u.count('@'),
                u.count('?'),
                u.count('='),
                u.count('.'),
                int(u.startswith("https")),
                int(u.count("//") > 1)
            ]
            for u in urls
        ])
        return csr_matrix(feats)

class GatingNetwork(nn.Module):
    def __init__(self, input_size=8, hidden_size=64, num_experts=2):
        super(GatingNetwork, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, num_experts)
        self.softmax = nn.Softmax(dim=1)
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        weights = self.softmax(x)
        return weights

# Phrase dictionary
phrase_dict = {
    'urgent': 0.3,
    'verify account': 0.5,
    'suspended': 0.4,
    'click here': 0.3,
    'confirm your': 0.4,
    'congratulations': 0.3,
    'winner': 0.4,
    'limited time': 0.3,
    'act now': 0.3,
    'security alert': 0.5,
    'claim': 0.3,
    'prize': 0.3,
    'free': 0.2,
    'bonus': 0.2,
}

def preprocess_text(text):
    if pd.isna(text) or text == "":
        return ""
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def calculate_phrase_score(text, phrase_dict):
    if not text:
        return 0.0
    text_lower = text.lower()
    score = 0.0
    for phrase, weight in phrase_dict.items():
        if phrase in text_lower:
            score += weight
    return min(score, 1.0)

def extract_gating_features(text, url, phrase_score):
    url_present = 1 if (url and not pd.isna(url) and url != "") else 0
    message_length = len(text.split()) if text else 0
    emoji_count = len(re.findall(r'[^\w\s,]', text)) if text else 0
    hashtag_count = text.count('#') if text else 0
    url_count = len(re.findall(r'http\S+', text)) if text else 0
    
    if text and len(text) > 0:
        capital_ratio = sum(1 for c in text if c.isupper()) / len(text)
    else:
        capital_ratio = 0.0
    
    embedding_summary = 0.0
    
    features = np.array([
        url_present,
        phrase_score,
        message_length,
        emoji_count,
        hashtag_count,
        url_count,
        capital_ratio,
        embedding_summary
    ], dtype=np.float32)
    
    return features

# ============================================================================
# UNIT TEST SUITE
# ============================================================================

class TestURLFeatures(unittest.TestCase):
    """Test URLFeatures transformer"""
    
    def setUp(self):
        self.url_features = URLFeatures()
    
    def test_url_feature_extraction_basic(self):
        """Test basic URL feature extraction"""
        urls = ["https://example.com"]
        features = self.url_features.transform(urls).toarray()
        
        # Check shape
        self.assertEqual(features.shape, (1, 8))
        
        # Check HTTPS detection
        self.assertEqual(features[0][6], 1)  # starts with https
    
    def test_url_feature_extraction_phishing_indicators(self):
        """Test phishing URL indicators"""
        urls = ["http://paypa1-security.com/verify?id=123&token=abc"]
        features = self.url_features.transform(urls).toarray()
        
        # Check for suspicious patterns
        self.assertGreater(features[0][1], 0)  # Has dashes
        self.assertGreater(features[0][3], 0)  # Has question marks
        self.assertGreater(features[0][4], 1)  # Has equals signs
        self.assertGreater(features[0][5], 0)  # Has dots
    
    def test_url_feature_extraction_empty(self):
        """Test empty URL handling"""
        urls = [""]
        features = self.url_features.transform(urls).toarray()
        
        # Empty URL should have minimal features
        self.assertEqual(features[0][0], 0)  # Length is 0
        self.assertEqual(features[0][6], 0)  # Not HTTPS
    
    def test_url_feature_extraction_multiple(self):
        """Test multiple URLs"""
        urls = ["https://google.com", "http://phishing-site.com", ""]
        features = self.url_features.transform(urls).toarray()
        
        self.assertEqual(features.shape, (3, 8))
        self.assertEqual(features[0][6], 1)  # First is HTTPS
        self.assertEqual(features[1][6], 0)  # Second is HTTP
        self.assertEqual(features[2][0], 0)  # Third is empty

class TestGatingNetwork(unittest.TestCase):
    """Test Gating Network architecture"""
    
    def setUp(self):
        self.gating_net = GatingNetwork(input_size=8, hidden_size=64, num_experts=2)
        self.gating_net.eval()
    
    def test_gating_network_output_shape(self):
        """Test gating network output shape"""
        input_tensor = torch.randn(1, 8)
        output = self.gating_net(input_tensor)
        
        self.assertEqual(output.shape, (1, 2))
    
    def test_gating_network_weights_sum_to_one(self):
        """Test that expert weights sum to 1"""
        input_tensor = torch.randn(5, 8)
        weights = self.gating_net(input_tensor)
        
        # Each row should sum to 1 (softmax property)
        sums = weights.sum(dim=1)
        torch.testing.assert_close(sums, torch.ones(5), rtol=1e-5, atol=1e-5)
    
    def test_gating_network_weights_range(self):
        """Test that weights are in [0, 1]"""
        input_tensor = torch.randn(10, 8)
        weights = self.gating_net(input_tensor)
        
        self.assertTrue(torch.all(weights >= 0))
        self.assertTrue(torch.all(weights <= 1))
    
    def test_gating_network_batch_processing(self):
        """Test batch processing"""
        batch_sizes = [1, 16, 32, 128]
        for batch_size in batch_sizes:
            input_tensor = torch.randn(batch_size, 8)
            output = self.gating_net(input_tensor)
            self.assertEqual(output.shape, (batch_size, 2))

class TestPreprocessing(unittest.TestCase):
    """Test preprocessing functions"""
    
    def test_preprocess_text_basic(self):
        """Test basic text preprocessing"""
        text = "Click here http://phishing.com now!"
        processed = preprocess_text(text)
        
        self.assertNotIn("http://", processed)
        self.assertEqual(processed, "Click here now!")
    
    def test_preprocess_text_empty(self):
        """Test empty text handling"""
        self.assertEqual(preprocess_text(""), "")
        self.assertEqual(preprocess_text(None), "")
        self.assertEqual(preprocess_text(np.nan), "")
    
    def test_preprocess_text_whitespace(self):
        """Test excessive whitespace removal"""
        text = "URGENT    verify    account"
        processed = preprocess_text(text)
        
        self.assertEqual(processed, "URGENT verify account")
    
    def test_preprocess_text_multiple_urls(self):
        """Test multiple URL removal"""
        text = "Visit http://site1.com and http://site2.com"
        processed = preprocess_text(text)
        
        self.assertNotIn("http://", processed)
        self.assertEqual(processed, "Visit and")
    
    def test_calculate_phrase_score_empty(self):
        """Test phrase score for empty text"""
        score = calculate_phrase_score("", phrase_dict)
        self.assertEqual(score, 0.0)
    
    def test_calculate_phrase_score_single_phrase(self):
        """Test phrase score for single phishing phrase"""
        score = calculate_phrase_score("URGENT: verify account", phrase_dict)
        
        # Should detect 'urgent' (0.3) and 'verify account' (0.5)
        self.assertEqual(score, 0.8)  # 0.3 + 0.5
    
    def test_calculate_phrase_score_multiple_phrases(self):
        """Test phrase score for multiple phrases"""
        text = "URGENT winner! Act now to claim your prize!"
        score = calculate_phrase_score(text, phrase_dict)
        
        # Should cap at 1.0 (urgent 0.3 + winner 0.4 + act now 0.3 + claim 0.3 + prize 0.3 = 1.6 → capped to 1.0)
        self.assertEqual(score, 1.0)
    
    def test_calculate_phrase_score_safe_text(self):
        """Test phrase score for safe text"""
        text = "Meeting scheduled for tomorrow at 2pm"
        score = calculate_phrase_score(text, phrase_dict)
        
        self.assertEqual(score, 0.0)

class TestFeatureExtraction(unittest.TestCase):
    """Test gating feature extraction"""
    
    def test_extract_gating_features_shape(self):
        """Test feature vector shape"""
        text = "Test message"
        url = "http://example.com"
        phrase_score = 0.5
        
        features = extract_gating_features(text, url, phrase_score)
        
        self.assertEqual(features.shape, (8,))
    
    def test_extract_gating_features_url_present(self):
        """Test URL presence detection"""
        features = extract_gating_features("Test", "http://site.com", 0.0)
        self.assertEqual(features[0], 1)  # URL present
        
        features = extract_gating_features("Test", "", 0.0)
        self.assertEqual(features[0], 0)  # URL absent
    
    def test_extract_gating_features_message_length(self):
        """Test message length calculation"""
        text = "This is a test message"
        features = extract_gating_features(text, "", 0.0)
        
        self.assertEqual(features[2], 5)  # 5 words
    
    def test_extract_gating_features_hashtags(self):
        """Test hashtag counting"""
        text = "#urgent #phishing #scam"
        features = extract_gating_features(text, "", 0.0)
        
        self.assertEqual(features[4], 3)  # 3 hashtags
    
    def test_extract_gating_features_capital_ratio(self):
        """Test capital letter ratio"""
        text = "URGENT"
        features = extract_gating_features(text, "", 0.0)
        
        self.assertEqual(features[6], 1.0)  # 100% capitals
        
        text = "urgent"
        features = extract_gating_features(text, "", 0.0)
        
        self.assertEqual(features[6], 0.0)  # 0% capitals
    
    def test_extract_gating_features_empty_text(self):
        """Test empty text handling"""
        features = extract_gating_features("", "", 0.0)
        
        self.assertEqual(features[2], 0)  # 0 words
        self.assertEqual(features[6], 0.0)  # 0 capital ratio

class TestEdgeCases(unittest.TestCase):
    """Test edge cases and error handling"""
    
    def test_very_long_text(self):
        """Test handling of very long text"""
        text = "word " * 1000  # 1000 words
        features = extract_gating_features(text, "", 0.0)
        
        self.assertEqual(features[2], 1000)
    
    def test_special_characters(self):
        """Test special character handling"""
        text = "🚨🚨🚨 URGENT!!! @@@"
        features = extract_gating_features(text, "", 0.0)
        
        # Should count special characters
        self.assertGreater(features[3], 0)
    
    def test_unicode_text(self):
        """Test Unicode text handling"""
        text = "긴급! 계정을 확인하세요"
        processed = preprocess_text(text)
        
        # Should handle Unicode without crashing
        self.assertIsInstance(processed, str)
    
    def test_malformed_urls(self):
        """Test malformed URL handling"""
        urls = [
            "htp://broken.com",
            "://noprotocol.com",
            "justtext",
            ""
        ]
        
        for url in urls:
            url_features = URLFeatures()
            features = url_features.transform([url]).toarray()
            # Should not crash
            self.assertEqual(features.shape, (1, 8))

class TestIntegration(unittest.TestCase):
    """Integration tests for the complete pipeline"""
    
    def test_phishing_detection_pipeline_phishing_text(self):
        """Test complete pipeline with phishing text"""
        # Updated text to clearly trigger multiple phishing indicators
        text = "URGENT! Your account will be SUSPENDED. Click here: http://paypa1.com/verify?id=123"
        
        url_match = re.findall(r'http[s]?://[^\s]+', text)
        
        if url_match:
            url = url_match[0]
            clean_text = re.sub(r'http\S+', '', text).strip()
        else:
            url = ""
            clean_text = text
        
        # Process through pipeline
        processed_text = preprocess_text(clean_text)
        phrase_score = calculate_phrase_score(processed_text, phrase_dict)
        features = extract_gating_features(processed_text, url, phrase_score)
        
        # Should match: "urgent" (0.3) + "suspended" (0.4) + "click here" (0.3) = 1.0 (capped)
        self.assertEqual(phrase_score, 1.0)  # Capped at maximum
        
        self.assertEqual(features[0], 1)  # URL present
        self.assertGreater(features[6], 0)  # Some capital letters
    
    def test_phishing_detection_pipeline_safe_text(self):
        """Test complete pipeline with safe text"""
        text = "Meeting scheduled for tomorrow at 2pm"
        
        processed_text = preprocess_text(text)
        phrase_score = calculate_phrase_score(processed_text, phrase_dict)
        features = extract_gating_features(processed_text, "", phrase_score)
        
        # Verify safe indicators
        self.assertEqual(phrase_score, 0.0)  # No phishing phrases
        self.assertEqual(features[0], 0)  # No URL
    
    def test_batch_processing_consistency(self):
        """Test that batch processing gives consistent results"""
        texts = [
            "URGENT verify account",  # Should score 0.8 (0.3 + 0.5)
            "Meeting tomorrow",
            "Claim your prize"  # "claim" = 0.3, "prize" = 0.3, total = 0.6
        ]
        
        scores = [calculate_phrase_score(t, phrase_dict) for t in texts]
        
        # Scores should be consistent
        self.assertEqual(scores[0], 0.8)  # Phishing
        self.assertEqual(scores[1], 0)    # Safe
        self.assertEqual(scores[2], 0.6)  # Phishing

# ============================================================================
# PERFORMANCE TESTS
# ============================================================================

class TestPerformance(unittest.TestCase):
    """Test performance metrics"""
    
    def test_preprocessing_speed(self):
        """Test preprocessing speed on 1000 samples"""
        import time
        
        texts = ["URGENT! Click here http://phishing.com"] * 1000
        
        start = time.perf_counter()
        for text in texts:
            _ = preprocess_text(text)
        elapsed = time.perf_counter() - start
        
        # Should process 1000 texts in under 1 second
        self.assertLess(elapsed, 1.0)
    
    def test_feature_extraction_speed(self):
        """Test feature extraction speed"""
        import time
        
        text = "URGENT! Verify account"
        url = "http://phishing.com"
        
        start = time.perf_counter()
        for _ in range(1000):
            _ = extract_gating_features(text, url, 0.5)
        elapsed = time.perf_counter() - start
        
        # Should extract 1000 feature sets in under 0.5 seconds
        self.assertLess(elapsed, 0.5)
    
    def test_url_feature_transformation_speed(self):
        """Test URL feature transformation speed"""
        import time
        
        url_features = URLFeatures()
        urls = ["http://example.com"] * 1000
        
        start = time.perf_counter()
        _ = url_features.transform(urls)
        elapsed = time.perf_counter() - start
        
        # Should transform 1000 URLs in under 0.1 seconds
        self.assertLess(elapsed, 0.1)

# ============================================================================
# TEST RUNNER
# ============================================================================

def run_tests():
    """Run all tests with detailed output"""
    
    print("="*80)
    print("RUNNING COMPREHENSIVE UNIT TESTS FOR MOE SYSTEM")
    print("="*80)
    
    # Create test suite
    loader = unittest.TestLoader()
    suite = unittest.TestSuite()
    
    # Add all test classes
    test_classes = [
        TestURLFeatures,
        TestGatingNetwork,
        TestPreprocessing,
        TestFeatureExtraction,
        TestEdgeCases,
        TestIntegration,
        TestPerformance
    ]
    
    for test_class in test_classes:
        tests = loader.loadTestsFromTestCase(test_class)
        suite.addTests(tests)
    
    # Run tests with verbose output
    runner = unittest.TextTestRunner(verbosity=2)
    result = runner.run(suite)
    
    # Print summary
    print("\n" + "="*80)
    print("TEST SUMMARY")
    print(f"Tests Run: {result.testsRun}")
    print(f"Successes: {result.testsRun - len(result.failures) - len(result.errors)}")
    print(f"Failures: {len(result.failures)}")
    print(f"Errors: {len(result.errors)}")
    
    if result.wasSuccessful():
        print("\n ALL TESTS PASSED!")
    else:
        print("\n SOME TESTS FAILED")
    
    print("="*80)
    
    return result

# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":
    run_tests()

test_url_feature_extraction_basic (__main__.TestURLFeatures.test_url_feature_extraction_basic)
Test basic URL feature extraction ... ok
test_url_feature_extraction_empty (__main__.TestURLFeatures.test_url_feature_extraction_empty)
Test empty URL handling ... ok
test_url_feature_extraction_multiple (__main__.TestURLFeatures.test_url_feature_extraction_multiple)
Test multiple URLs ... ok
test_url_feature_extraction_phishing_indicators (__main__.TestURLFeatures.test_url_feature_extraction_phishing_indicators)
Test phishing URL indicators ... ok
test_gating_network_batch_processing (__main__.TestGatingNetwork.test_gating_network_batch_processing)
Test batch processing ... ok
test_gating_network_output_shape (__main__.TestGatingNetwork.test_gating_network_output_shape)
Test gating network output shape ... ok
test_gating_network_weights_range (__main__.TestGatingNetwork.test_gating_network_weights_range)
Test that weights are in [0, 1] ... ok
test_gating_network_weights_sum_to_one (__main__.

RUNNING COMPREHENSIVE UNIT TESTS FOR MOE SYSTEM

TEST SUMMARY
Tests Run: 32
Successes: 32
Failures: 0
Errors: 0

 ALL TESTS PASSED!
